In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q "/content/drive/MyDrive/2413-Sanket-Naik-Dissertation-2025-26/KaggleDataset/ISICDataset.zip" -d "/content/data/"
print("Extrated files")

Extrated files


In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision import models

import albumentations as A
from albumentations.pytorch import ToTensorV2

from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, matthews_corrcoef, cohen_kappa_score,
    roc_curve
)

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CONFIG = {
    "batch_size": 32,
    "epochs": 12,
    "lr": 1e-4,
    "img_size": 224
}

INDEX_PATH = "/content/drive/MyDrive/2413-Sanket-Naik-Dissertation-2025-26/Implimentations/final_dataset_index.csv"
BASE_DIR = "/content/data/train-image/image"
MODEL_PATH = "/content/drive/MyDrive/2413-Sanket-Naik-Dissertation-2025-26/Implimentations/Fusion resnet18/fusion_resnet18.pth"

In [ ]:
def remove_hair(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(1,(17,17))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, thresh = cv2.threshold(blackhat,10,255,cv2.THRESH_BINARY)
    return cv2.inpaint(image, thresh, 1, cv2.INPAINT_TELEA)

In [ ]:
def preprocess_metadata(df):
    df = df.copy()

    df["age_approx"] = df["age_approx"].fillna(df["age_approx"].median())

    df["sex"] = df["sex"].map({"male": 0, "female": 1})
    df["sex"] = df["sex"].fillna(0)

    df = pd.get_dummies(df, columns=["anatom_site_general"])

    return df

In [ ]:
class ISICDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

        self.meta_cols = [c for c in df.columns if c.startswith("anatom_site_general")] + ["age_approx", "sex"]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = os.path.join(BASE_DIR, os.path.basename(row["path"]))
        image = np.array(Image.open(img_path).convert("RGB"))
        image = remove_hair(image)

        if self.transform:
            image = self.transform(image=image)["image"]

        meta = torch.tensor(row[self.meta_cols].values.astype(np.float32))
        label = torch.tensor(row["target"], dtype=torch.float32)

        return image, meta, label

In [ ]:
train_tf = A.Compose([
    A.Resize(224,224),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=25,p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.Normalize(),
    ToTensorV2()
])

val_tf = A.Compose([
    A.Resize(224,224),
    A.Normalize(),
    ToTensorV2()
])

In [ ]:
df = pd.read_csv(INDEX_PATH)
df = preprocess_metadata(df)

train_df = df[df["split"]=="train"]
val_df   = df[df["split"]=="val"]
test_df  = df[df["split"]=="test"]

train_dataset = ISICDataset(train_df, train_tf)
val_dataset   = ISICDataset(val_df, val_tf)
test_dataset  = ISICDataset(test_df, val_tf)

In [ ]:
pos = train_df["target"].sum()
neg = len(train_df) - pos

weights = train_df["target"].apply(
    lambda x: 1/pos if x==1 else 1/neg
)

sampler = WeightedRandomSampler(weights.values, len(weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], sampler=sampler)
val_loader   = DataLoader(val_dataset, batch_size=CONFIG["batch_size"], shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=CONFIG["batch_size"], shuffle=False)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2, alpha=0.75):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs = torch.sigmoid(logits)
        pt = torch.where(targets==1, probs, 1-probs)
        return (self.alpha * (1-pt)**self.gamma * bce).mean()

criterion = FocalLoss()

In [ ]:
class FusionModel(nn.Module):
    def __init__(self, meta_dim):
        super().__init__()

        self.backbone = models.resnet18(weights="IMAGENET1K_V1")
        self.backbone.fc = nn.Identity()

        self.meta_net = nn.Sequential(
            nn.Linear(meta_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3)
        )

        self.classifier = nn.Sequential(
            nn.Linear(512 + 64, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 1)
        )

    def forward(self, image, meta):
        img_feat = self.backbone(image)
        meta_feat = self.meta_net(meta)

        x = torch.cat([img_feat, meta_feat], dim=1)
        return self.classifier(x)

In [ ]:
meta_dim = len(train_dataset.meta_cols)

model = FusionModel(meta_dim).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=CONFIG["lr"])

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 187MB/s]


In [ ]:
best_auc = 0

for epoch in range(CONFIG["epochs"]):
    model.train()
    running_loss = 0

    for images, meta, labels in tqdm(train_loader):
        images = images.to(DEVICE)
        meta = meta.to(DEVICE)
        labels = labels.unsqueeze(1).to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images, meta)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # VALIDATION
    model.eval()
    y_true, y_prob = [], []

    with torch.no_grad():
        for images, meta, labels in val_loader:
            images = images.to(DEVICE)
            meta = meta.to(DEVICE)

            probs = torch.sigmoid(model(images, meta)).cpu().numpy().flatten()

            y_prob.extend(probs)
            y_true.extend(labels.numpy())

    auc = roc_auc_score(y_true, y_prob)
    print(f"Epoch {epoch+1} | Loss {running_loss:.4f} | Val AUC {auc:.4f}")

    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), MODEL_PATH)

100%|██████████| 9330/9330 [12:10<00:00, 12.77it/s]


Epoch 1 | Loss 137.4669 | Val AUC 0.9146


100%|██████████| 9330/9330 [12:07<00:00, 12.82it/s]


Epoch 2 | Loss 44.2662 | Val AUC 0.9182


100%|██████████| 9330/9330 [12:07<00:00, 12.83it/s]


...

Epoch 10 | Loss 9.8431 | Val AUC 0.8472


100%|██████████| 9330/9330 [12:07<00:00, 12.83it/s]


Epoch 11 | Loss 8.2050 | Val AUC 0.8807


100%|██████████| 9330/9330 [12:08<00:00, 12.81it/s]


Epoch 12 | Loss 9.3889 | Val AUC 0.8696


In [ ]:
fpr, tpr, thresholds = roc_curve(y_true, y_prob)

specificity = 1 - fpr
idx = np.where(specificity >= 0.90)[0][-1]
best_threshold = thresholds[idx]

print("Threshold:", best_threshold)

Threshold: 0.0024469155


In [ ]:
model.load_state_dict(torch.load(MODEL_PATH))
model.eval()

y_true, y_pred, y_prob = [], [], []

with torch.no_grad():
    for images, meta, labels in test_loader:
        images = images.to(DEVICE)
        meta = meta.to(DEVICE)

        probs = torch.sigmoid(model(images, meta)).cpu().numpy().flatten()
        preds = (probs >= best_threshold).astype(int)

        y_prob.extend(probs)
        y_pred.extend(preds)
        y_true.extend(labels.numpy())

acc = accuracy_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob)
mcc = matthews_corrcoef(y_true, y_pred)
kappa = cohen_kappa_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print("\nFUSION RESULTS")
print("Accuracy:", acc)
print("Recall:", rec)
print("F1:", f1)
print("AUROC:", auc)
print("MCC:", mcc)
print("Kappa:", kappa)
print("Confusion Matrix:\n", cm)


FUSION RESULTS
Accuracy: 0.8605688561489666
Recall: 0.7538461538461538
F1: 0.014178240740740741
AUROC: 0.8842334880568669
MCC: 0.0645293691950336
Kappa: 0.011573635364605761
Confusion Matrix:
 [[42007  6798]
 [   16    49]]
